In [2]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [3]:
%load_ext autoreload
%autoreload 2

# Model definition

In [4]:
import hydra
from omegaconf import OmegaConf
import torch

conf = OmegaConf.load('config/whisper.yaml')

In [5]:
from ptls.nn import TrxEncoder

trx_encoder_params = conf['seq_encoder']['trx_encoder']
trx_encoder = TrxEncoder(**trx_encoder_params)

In [6]:
from ptls.nn import RnnEncoder

seq_encoder_params = {'type': 'gru', 'hidden_size': 1024, 'bidir': False}
rnn_seq_encoder = RnnEncoder(input_size=768, is_reduce_sequence=True, **seq_encoder_params)

In [7]:
from romashka.transactions_qa.layers.layers import LambdaLayer
from romashka.transactions_qa.utils import zero_function
from ptls.data_load.padded_batch import PaddedBatch

class WhisperSeqEncoder(torch.nn.Module):
    
    def __init__(self, trx_encoder, seq_encoder, whisper_encoder):
        super().__init__()
        self.trx_encoder = trx_encoder
        self.seq_encoder = seq_encoder
        self.whisper_encoder = whisper_encoder
        
    @property
    def is_reduce_sequence(self):
        return self.seq_encoder.is_reduce_sequence

    @is_reduce_sequence.setter
    def is_reduce_sequence(self, value):
        self.seq_encoder.is_reduce_sequence = value

    @property
    def category_max_size(self):
        return self.trx_encoder.category_max_size

    @property
    def category_names(self):
        return self.trx_encoder.category_names

    @property
    def embedding_size(self):
        return self.seq_encoder.embedding_size

    def forward(self, x, h_0=None):
        x = self.trx_encoder(x)
        length = x._length
        x = self.whisper_encoder(inputs_embeds=x.payload, attention_mask=x.seq_len_mask).last_hidden_state
        x = PaddedBatch(payload=x, length=length)
        x = self.seq_encoder(x, h_0)
        return x
    
from transformers import AutoModel, AutoConfig


whisper_encoder = AutoModel.from_pretrained('openai/whisper-small').decoder
whisper_encoder.embed_positions = LambdaLayer(zero_function)
seq_encoder = WhisperSeqEncoder(trx_encoder, rnn_seq_encoder, whisper_encoder)

# Finetune

In [8]:
from glob import glob
from ptls.data_load.iterable_processing_dataset import IterableProcessingDataset
from ptls.data_load.iterable_processing.target_move import TargetMove
from ptls.data_load.iterable_processing.target_empty_filter import TargetEmptyFilter
from ptls.data_load import padded_collate, padded_collate_wo_target
from ptls.data_load.iterable_processing.to_torch_tensor import ToTorch
from ptls.data_load.datasets import MemoryMapDataset
from tqdm.auto import tqdm

from ptls.data_load import IterableChain
from ptls.data_load.iterable_processing import SeqLenFilter
from ptls.data_load.datasets.parquet_dataset import ParquetDataset, ParquetFiles
from ptls.data_load.utils import collate_feature_dict


from ptls.frames import PtlsDataModule

train_data = glob('data/train_transactions_clipped_bucket_means.parquet')
valid_data = glob('data/valid_transactions_clipped_bucket_means.parquet')

feature_cols = list(seq_encoder.trx_encoder.embeddings.keys()) + \
               list(seq_encoder.trx_encoder.numeric_values.keys())

target_cols = ['mcc', 'amnt', 'hour_diff']

dataset_conf = {
    'min_seq_len':25,
    }



class SeqToTargetMultiheadDataset(torch.utils.data.Dataset):
    def __init__(self,
                 data,
                 feature_cols,
                 target_cols,
                 target_dtype=None,
                 *args, **kwargs,
                 ):
        super().__init__(*args, **kwargs)

        self.data = data
        self.feature_cols = feature_cols
        self.target_cols = target_cols
        
        if type(target_dtype) is str:
            self.target_dtype = getattr(torch, target_dtype)
        else:
            self.target_dtype = target_dtype

    def __len__(self):
        return len(self.data)

    def __getitem__(self, item):
        feature_arrays = self.data[item]
        return feature_arrays

    def __iter__(self):
        for feature_arrays in self.data:
            yield feature_arrays


    def collate_fn(self, batch):
        
        targets = []
        values = []
        
        for target_col in target_cols:
            targets.append(torch.tensor([rec[target_col][-1] for rec in batch]).to(self.target_dtype[target_col]))
        
        for rec in batch:
            values.append({k: v[:-1] for k, v in rec.items() if k in feature_cols})
    
        return padded_collate_wo_target(values), targets

process = IterableChain(
            SeqLenFilter(min_seq_len=dataset_conf['min_seq_len']),
            ToTorch()
            )
    
def get_dataset(data):
    ds = MemoryMapDataset(ParquetDataset(data, post_processing=process))
    return SeqToTargetMultiheadDataset(ds, feature_cols, target_cols, target_dtype = {'mcc': torch.long, 'amnt': torch.float, 'hour_diff': torch.float})

train_ds = get_dataset(train_data)
valid_ds = get_dataset(valid_data)

dm = PtlsDataModule(
    train_data=train_ds,
    valid_data=valid_ds,
    train_num_workers=4,
    train_batch_size=64)

[2024-08-06 15:41:33,142] [INFO] [real_accelerator.py:161:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/morlov/.local/share/virtualenvs/pytorch-lifestream-1iBTwtzi/lib/python3.8/site-packages/ptls/data_load/datasets/parquet_dataset.py:106: UserWarning: `post_processing` parameter is deprecated, use `i_filters`
  warnings.warn('`post_processing` parameter is deprecated, use `i_filters`')


In [9]:
import logging
from copy import deepcopy
from typing import List

import pandas as pd
import pytorch_lightning as pl
import torch
import torchmetrics
from omegaconf import DictConfig

from ptls.data_load.padded_batch import PaddedBatch

logger = logging.getLogger(__name__)


class SequenceToTargetMultihead(pl.LightningModule):


    def __init__(self,
                 seq_encoder: torch.nn.Module,
                 heads: List[torch.nn.Module],
                 losses: List[torch.nn.Module],
                 metric_list: torchmetrics.Metric=None,
                 optimizer_partial=None,
                 lr_scheduler_partial=None,
                 pretrained_lr=None,
                 train_update_n_steps=None,
                 ):
        super().__init__()

        self.save_hyperparameters(ignore=[
            'seq_encoder', 'heads', 'losses', 'metric_list', 'optimizer_partial', 'lr_scheduler_partial'])

        self.seq_encoder = seq_encoder
        self.heads = heads
        self.losses = losses
        self.n_heads = len(heads)

        self.optimizer_partial = optimizer_partial
        self.lr_scheduler_partial = lr_scheduler_partial

    def forward(self, x):
        x = self.seq_encoder(x)
        xs = [head(x) for head in self.heads]
        return xs

    def training_step(self, batch, _):
        x, y = batch
        y_hs = self(x)
        loss = sum([loss(y_hs[i], y[i]) for i, loss in enumerate(self.losses)])
        self.log('loss', loss)
        return loss

    def validation_step(self, batch, _):
        x, y = batch
        y_hs = self(x)
        loss = sum([loss(y_hs[i], y[i]) for i, loss in enumerate(self.losses)])
        self.log('val_loss', loss)


    def configure_optimizers(self):
        if self.hparams.pretrained_lr is not None:
            if self.hparams.pretrained_lr == 'freeze':
                for p in self.seq_encoder.parameters():
                    p.requires_grad = False
                parameters = self.parameters()
            else:
                parameters = [
                    {'params': self.seq_encoder.parameters(), 'lr': self.hparams.pretrained_lr},
                ] + [{'params': head.parameters()} for head in self.heads]
        else:
            parameters = self.parameters()

        optimizer = self.optimizer_partial(parameters)
        scheduler = self.lr_scheduler_partial(optimizer)
        return [optimizer], [scheduler]



In [10]:
from functools import partial
import torch
import torchmetrics
from ptls.nn import Head


head_mcc = Head(input_size=seq_encoder.embedding_size, 
                use_batch_norm=True,
                hidden_layers_sizes=[128],
                objective='classification',
                num_classes=109).to('cuda:0')

head_amnt = Head(input_size=seq_encoder.embedding_size, 
                 use_batch_norm=True,
                 hidden_layers_sizes=[128],
                 objective='softplus').to('cuda:0')

head_hour_diff = Head(input_size=seq_encoder.embedding_size, 
                      use_batch_norm=True,
                      hidden_layers_sizes=[128],
                      objective='softplus').to('cuda:0')

model_multihead = SequenceToTargetMultihead(
    seq_encoder=seq_encoder,
    heads=[head_mcc, head_amnt, head_hour_diff],
    losses=[torch.nn.NLLLoss(), torch.nn.L1Loss(), torch.nn.L1Loss()],
    metric_list=torchmetrics.Accuracy(task='binary'),
    pretrained_lr=3e-4, # 0.00001,
    optimizer_partial=partial(torch.optim.Adam, lr=3e-4), # , weight_decay=1e-5
    lr_scheduler_partial=partial(torch.optim.lr_scheduler.StepLR, step_size=1, gamma=0.9),
)

In [11]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import LearningRateMonitor
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger

trainer_params = conf.trainer
print(OmegaConf.to_yaml(trainer_params))



trainer_params = conf.trainer
trainer_params['max_epochs']  = 15
callbacks = [ModelCheckpoint(every_n_epochs=5, save_top_k=-1), LearningRateMonitor(logging_interval='step')]
logger = TensorBoardLogger(save_dir='lightning_logs', name='whisper-next-event-bucket-mean')
trainer = pl.Trainer(**trainer_params, callbacks=callbacks, logger=logger)

GPU available: True, used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


gpus: 1
auto_select_gpus: false
max_epochs: 30
deterministic: true



In [ ]:
%%time
print(f'logger.version = {trainer.logger.version}')
trainer.fit(model_multihead, dm)
print(trainer.logged_metrics)

logger.version = 1


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name        | Type              | Params
--------------------------------------------------
0 | seq_encoder | WhisperSeqEncoder | 158 M 
--------------------------------------------------
158 M     Trainable params
0         Non-trainable params
158 M     Total params
635.623   Total estimated model params size (MB)


Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

In [ ]:
torch.save(model_multihead.state_dict(), "models/whisper-e2e-nep-bucket-means.pt")

# Infernece

In [ ]:
model_multihead.load_state_dict(torch.load("models/whisper-e2e-nep-bucket-means.pt"))

In [30]:
# %%time
import tqdm

def inference(model, dl, device='cuda:0'):
    
    model.to(device)
    X = []
    for batch in tqdm.tqdm(dl):
        with torch.no_grad():
            features = batch[0]
            targets = [t.unsqueeze(dim=1).to(device) for t in batch[1]]
            x = model(features.to(device))
            mcc = torch.argmax(x[0], dim=1, keepdim=True)
            amnt = x[1].unsqueeze(dim=1)
            hour_diff = x[2].unsqueeze(dim=1)
            predicted = [mcc, amnt, hour_diff]
            X += [torch.cat(predicted + targets, dim=1)]
    return X


valid_dl = torch.utils.data.DataLoader(dataset=valid_ds, 
                                       collate_fn=valid_ds.collate_fn,
                                       num_workers=8,
                                       batch_size=128)

In [31]:
preds = torch.vstack(inference(model_multihead, valid_dl)).cpu().numpy()

100%|█████████████████████████████████████████| 203/203 [01:14<00:00,  2.73it/s]


In [32]:
import numpy as np

df_valid = pd.DataFrame(preds, columns = ['predicted_mcc', 'predicted_amnt', 'predicted_hour_diff', 'mcc', 'amnt', 'hour_diff'])
df_valid.head()

,predicted_mcc,predicted_amnt,predicted_hour_diff,mcc,amnt,hour_diff
0,2.0,0.409114,35.001282,1.0,0.123295,0.5
1,2.0,0.409114,35.001282,14.0,0.123295,35.0
2,2.0,0.409114,35.001282,1.0,0.261539,6.5
3,2.0,0.409114,35.001282,2.0,1.000000,20.0
4,2.0,0.409114,35.001282,35.0,0.409312,71.0


## MCC

In [25]:
from sklearn.metrics import accuracy_score, f1_score

print("Accuracy:", {accuracy_score(df_valid['mcc'],  df_valid['predicted_mcc'])})
print("F1:", {f1_score(df_valid['mcc'],  df_valid['predicted_mcc'], average='weighted')})

Accuracy: {0.3881890673754767}
F1: {0.21710407547689414}


# amnt

In [27]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("MAE AMNT:", {mean_absolute_error(df_valid['amnt'],  df_valid['predicted_amnt'])})
print("MSE AMNT:", {mean_squared_error(df_valid['amnt'],  df_valid['predicted_amnt'])})

MAE AMNT: {0.20585918}
MSE AMNT: {0.09492184}


# hour_diff

In [29]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("MAE:", {mean_absolute_error(df_valid['hour_diff'],  df_valid['predicted_hour_diff'])})
print("MSE:", {mean_squared_error(df_valid['hour_diff'],  df_valid['predicted_hour_diff'])})

MAE: {36.33046}
MSE: {1699.1664}


In [15]:
train_dl = torch.utils.data.DataLoader(dataset=train_ds, 
                                       collate_fn=train_ds.collate_fn,
                                       num_workers=8,
                                       batch_size=4)

In [20]:
x, target = next(iter(train_dl))

In [21]:
target

[tensor([2, 2, 2, 2]),
 tensor([1.0000, 1.0000, 0.4093, 1.0000]),
 tensor([71.0000,  3.0000, 95.0000,  0.5000])]

In [19]:
x.payload

{'amnt': tensor([[1.0000, 0.3782, 0.4568, 0.2876, 0.3314, 1.0000, 1.0000, 0.3782, 0.4568,
          1.0000, 0.3540, 0.3314, 0.4093, 0.3314, 0.3540, 1.0000, 0.3782, 1.0000,
          0.3782, 1.0000, 0.4568, 0.3314, 0.2615, 1.0000, 0.4093, 1.0000, 0.3782,
          1.0000, 0.4568, 1.0000, 0.4568, 0.4093, 0.2876, 0.3314, 1.0000, 0.4093,
          0.3782, 0.4568, 0.3314, 0.4093, 1.0000, 0.4568, 0.4568, 0.4568, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.4568, 0.3540, 1.0000, 1.0000, 0.3782, 0.4568, 0.4568, 0.3782, 0.4093,
          0.3782, 1.0000, 1.0000, 0.4568, 0.4093, 1.0000, 0.3540, 1.0000, 0.4568,
          1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
          0.4568, 0.4568, 1.0000, 1.0000, 1.0000, 0.4093, 0.2615, 1.0000, 0.4093,
          0.2615, 0.4568, 0.4093, 0.4093, 1.0000, 0.3314, 1.0000, 0.3782, 0.3095,
          1.0000, 0.3540, 0.3095, 0.3782],
         [0.4093, 0.3540, 1.0000, 0.4093, 0.4568, 1.0000, 0.4568, 1.0000, 0.4568,
    